# Import Required libraries

In [41]:
# Python basic packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# sklearn packages
from sklearn.model_selection import train_test_split


# tensorflow packages
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM,
                                    Dropout)
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint)
from tensorflow.keras.losses import (SparseCategoricalCrossentropy,
                                     CategoricalCrossentropy)                                         

# Load Train, Validation and Test Dataset

In [2]:
X_train = pd.read_csv("./processed_data/RMS_X_train_features.csv").drop(columns = ["SubjectID", "Motor_label"])
X_train["Label"] = X_train["Label"] - 1
X_test = pd.read_csv("./processed_data/RMS_X_test_features.csv").drop(columns = ["SubjectID", "Motor_label"])
X_test["Label"] = X_test["Label"] - 1

In [3]:
X_train, X_val, y_train, y_val = train_test_split(X_train.drop("Label", axis = 1), X_train["Label"], train_size=0.8, random_state=42)
print(f"Shape of X_train : {X_train.shape}")
print(f"Shape of X_train : {X_val.shape}")
print(f"Shape of X_train : {y_train.shape}")
print(f"Shape of X_train : {y_val.shape}")

Shape of X_train : (268, 15)
Shape of X_train : (68, 15)
Shape of X_train : (268,)
Shape of X_train : (68,)


# Create tf.Dataset

In [4]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, to_categorical(y_train))).batch(64)
validation_dataset = tf.data.Dataset.from_tensor_slices((X_val, to_categorical(y_val))).batch(64)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test.drop("Label", axis = 1), X_test["Label"])).batch(64)

2025-08-03 10:22:49.172146: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-08-03 10:22:49.172221: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-08-03 10:22:49.172230: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.00 GB
I0000 00:00:1754234569.172243 1340138 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1754234569.172263 1340138 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


# Neural Network Model

#### Step 1: Baseline Model Architecture

In [5]:
class SimpleDLModel(Model):
    def __init__(self, num_classes):
        super(SimpleDLModel, self).__init__()
        self.dense1 = Dense(1024, activation='leaky_relu')
        self.dropout1 = Dropout(0.2)
        self.dense2 = Dense(512, activation='leaky_relu')
        self.dropout2 = Dropout(0.2)
        self.dense2 = Dense(256, activation='leaky_relu')
        self.dropout2 = Dropout(0.2)
        self.dense2 = Dense(128, activation='leaky_relu')
        self.dropout2 = Dropout(0.2)
        self.output_layer = Dense(num_classes, activation='softmax')

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dropout1(x)
        x = self.dense2(x)
        x = self.dropout2(x)
        return self.output_layer(x)
    
SimpleDLModel = SimpleDLModel(num_classes=7)   

#### Step 2: Compile the Model

In [6]:
SimpleDLModel.compile(optimizer=AdamW(learning_rate=0.001),
                      loss=CategoricalCrossentropy(from_logits=False),
                      metrics=['f1_score', 'accuracy'])

#### Step 3: Train the Model

In [7]:
SimpleDLModel.fit(train_dataset,
                 validation_data=validation_dataset,
                 epochs=100,
                 )

Epoch 1/100


/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'simple_dl_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
2025-08-03 10:22:49.490088: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - accuracy: 0.1448 - f1_score: 0.0683 - loss: 1.9726 - val_accuracy: 0.0882 - val_f1_score: 0.0232 - val_loss: 1.9807
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1767 - f1_score: 0.0579 - loss: 1.9470 - val_accuracy: 0.1176 - val_f1_score: 0.0301 - val_loss: 1.9726
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1494 - f1_score: 0.0371 - loss: 1.9621 - val_accuracy: 0.1471 - val_f1_score: 0.0640 - val_loss: 1.9610
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1538 - f1_score: 0.0675 - loss: 1.9633 - val_accuracy: 0.1618 - val_f1_score: 0.0398 - val_loss: 1.9537
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1404 - f1_score: 0.0352 - loss: 1.9583 - val_accuracy: 0.1618 - val_f1_score: 0.0398 - val_loss: 1.9531
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1417 - f1_score: 0.0375 - loss: 1.9524 - val_accuracy: 0.1029 - val_f1_score: 0.0443 - val_loss: 1.9557
Epoc

#### Step 4: Save the keras Model

In [ ]:
SimpleDLModel.save("./artifacts/base_model.keras")

# Prediction on test dataset

In [37]:
test_predictions= np.argmax(SimpleDLModel.predict(test_dataset), axis=1) 
test_predictions = pd.DataFrame(test_predictions, columns=["Predicted_Label"])

# Get the labels
all_labels = []
for _, y in test_dataset:
    all_labels.extend(y.numpy().tolist())
test_predictions["Label"] = all_labels

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


2025-08-03 10:41:45.275563: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [38]:
np.argmax(np.round(SimpleDLModel.predict(test_dataset)[0], 3), axis=0)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


np.int64(6)

#### Metrics and Evaluation

#### Metric 1: Confusion Matrix

In [33]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_predictions["Label"], test_predictions["Predicted_Label"])
cm

array([[2, 5, 1, 0, 2, 0, 2],
       [0, 7, 0, 0, 3, 0, 2],
       [1, 6, 1, 0, 2, 0, 2],
       [1, 6, 1, 0, 4, 0, 0],
       [0, 9, 0, 1, 1, 0, 1],
       [0, 4, 3, 1, 3, 0, 1],
       [1, 5, 1, 0, 4, 0, 1]])

#### Metric 2: Classification Report

In [43]:
from sklearn.metrics import classification_report
print(classification_report(test_predictions["Label"], test_predictions["Predicted_Label"]))

              precision    recall  f1-score   support

           0       0.40      0.17      0.24        12
           1       0.17      0.58      0.26        12
           2       0.14      0.08      0.11        12
           3       0.00      0.00      0.00        12
           4       0.05      0.08      0.06        12
           5       0.00      0.00      0.00        12
           6       0.11      0.08      0.10        12

    accuracy                           0.14        84
   macro avg       0.12      0.14      0.11        84
weighted avg       0.12      0.14      0.11        84

